<a href="https://colab.research.google.com/github/Tejaswini-08-2005/NLP/blob/main/NLP_Assignment_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import pandas as pd
df = pd.read_csv('/content/movie_review.csv.zip')
df = df.drop(columns=['sent_id','html_id','cv_tag','fold_id'])
df.head()

,text,tag
0,films adapted from comic books have had plenty...,pos
1,"for starters , it was created by alan moore ( ...",pos
2,to say moore and campbell thoroughly researche...,pos
3,"the book ( or "" graphic novel , "" if you will ...",pos
4,"in other words , don't dismiss this film becau...",pos


In [28]:
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [29]:
data = {
    "review": [
        "This movie is amazing and wonderful",
        "I really liked the acting and story",
        "The film was boring and slow",
        "Worst movie I have ever seen",
        "Fantastic plot and great performance",
        "Terrible movie and bad acting",
        "I enjoyed this movie a lot",
        "Not good very disappointing",
        "Excellent direction and music",
        "Waste of time"
    ],
    "sentiment": [
        "positive","positive","negative","negative","positive",
        "negative","positive","negative","positive","negative"
    ]
}

df = pd.DataFrame(data)
print(df.head())

                                 review sentiment
0   This movie is amazing and wonderful  positive
1   I really liked the acting and story  positive
2          The film was boring and slow  negative
3          Worst movie I have ever seen  negative
4  Fantastic plot and great performance  positive


In [30]:
# Text Preprocessing Pipeline
def nltk_preprocessing_pipeline(text):

    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags
    text = re.sub(r'#\w+', '', text)

    # Convert to lowercase
    text = text.lower()

    # Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )

    text = emoji_pattern.sub(r'', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Word Tokenization
    tokenized_words = word_tokenize(text)

    # Stopword Removal
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in tokenized_words if word not in stop_words]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

    # Join words
    clean_text = " ".join(lemmatized_words)

    return clean_text

df["text"] = df["review"].apply(nltk_preprocessing_pipeline)
df.head()

,review,sentiment,text
0,This movie is amazing and wonderful,positive,movie amazing wonderful
1,I really liked the acting and story,positive,really liked acting story
2,The film was boring and slow,negative,film boring slow
3,Worst movie I have ever seen,negative,worst movie ever seen
4,Fantastic plot and great performance,positive,fantastic plot great performance


In [31]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df["text"])

# Display Vocabulary Size
vocab_size = len(tfidf.vocabulary_)

print("\nVocabulary Size:", vocab_size)

# Show some vocabulary words
print("\nSample vocabulary words:")
print(list(tfidf.vocabulary_.keys())[:20])

# TF-IDF Matrix Shape
print("\nTF-IDF Matrix Shape:", X_tfidf.shape)


Vocabulary Size: 28

Sample vocabulary words:
['movie', 'amazing', 'wonderful', 'really', 'liked', 'acting', 'story', 'film', 'boring', 'slow', 'worst', 'ever', 'seen', 'fantastic', 'plot', 'great', 'performance', 'terrible', 'bad', 'enjoyed']

TF-IDF Matrix Shape: (10, 28)


In [23]:

#Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)


In [24]:
#Traning
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [32]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
#Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

#Classification Report
print(classification_report(y_test, y_pred))

#Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.0
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       0.0
    positive       0.00      0.00      0.00       2.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0

[[0 0]
 [2 0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_